# Forward Model Dataset Preparation

Bu notebook'ta node ve linker SMILES, point group ve topology
girdileri kullanılarak density, PLD ve LCD hedefleri için model
veri seti hazırlanmaktadır.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent
QMOF_CSV_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "qmof_database"
    / "qmof.csv"
)

df = pd.read_csv(QMOF_CSV_PATH, low_memory=False)

print("Ana veri seti:", df.shape)

Ana veri seti: (20372, 94)


In [2]:
selected_columns = {
    "qmof_id": "qmof_id",
    "info.mofid.smiles_nodes": "smiles_nodes",
    "info.mofid.smiles_linkers": "smiles_linkers",
    "info.symmetry.pointgroup": "point_group",
    "info.mofid.topology": "topology",
    "info.density": "density",
    "info.pld": "pld",
    "info.lcd": "lcd",
}

model_df = (
    df[list(selected_columns.keys())]
    .rename(columns=selected_columns)
    .copy()
)

print("Model veri seti:", model_df.shape)

display(model_df.head())

Model veri seti: (20372, 8)


,qmof_id,smiles_nodes,smiles_linkers,point_group,topology,density,pld,lcd
0,qmof-8a95c27,"['O', '[Ba]', '[Cu]']",['[O-]C=O'],-1,NaN,2.763246,0.68822,1.35480
1,qmof-019ba28,NaN,NaN,2/m,NaN,3.229952,1.18570,2.13507
2,qmof-830ed1c,['[Co]'],['[O-]C(=O)c1ccncc1'],m,rtl,1.557644,2.36128,4.21176
3,qmof-5bd4a24,['[Co]'],['[O-]C(=O)c1ccncc1'],2/m,rtl,1.616139,2.14542,3.27957
4,qmof-644aab4,['[Zn][Zn]'],"['[O-]C(=O)c1cccc(c1)c1nccs1', 'n1ccc(cc1)c1cc...",-1,NaN,1.596537,1.33452,2.03948


In [3]:
missing_summary = pd.DataFrame({
    "data_type": model_df.dtypes.astype(str),
    "null_count": model_df.isna().sum(),
    "null_percentage": (
        model_df.isna().mean() * 100
    ).round(2),
    "unique_count": model_df.nunique(dropna=True),
})

missing_summary = missing_summary.sort_values(
    by="null_percentage",
    ascending=False,
)

display(missing_summary)

,data_type,null_count,null_percentage,unique_count
topology,object,12472,61.22,211
smiles_linkers,object,2828,13.88,9852
smiles_nodes,object,2695,13.23,1796
qmof_id,object,0,0.00,20372
point_group,object,0,0.00,31
density,float64,0,0.00,20372
pld,float64,0,0.00,19495
lcd,float64,0,0.00,19899


In [4]:
target_columns = [
    "density",
    "pld",
    "lcd",
]

print("Hedef kolonlardaki eksik değerler:\n")
print(model_df[target_columns].isna().sum())

Hedef kolonlardaki eksik değerler:

density    0
pld        0
lcd        0
dtype: int64


In [5]:
input_columns = [
    "smiles_nodes",
    "smiles_linkers",
    "point_group",
    "topology",
]

input_missing_summary = pd.DataFrame({
    "null_count": model_df[input_columns].isna().sum(),
    "null_percentage": (
        model_df[input_columns].isna().mean() * 100
    ).round(2),
})

display(input_missing_summary)

,null_count,null_percentage
smiles_nodes,2695,13.23
smiles_linkers,2828,13.88
point_group,0,0.00
topology,12472,61.22


In [6]:
complete_input_mask = model_df[input_columns].notna().all(axis=1)

complete_input_count = complete_input_mask.sum()
incomplete_input_count = (~complete_input_mask).sum()

print("Tüm girdileri dolu kayıt:", complete_input_count)
print("En az bir girdisi eksik kayıt:", incomplete_input_count)
print(
    "Kullanılabilir oran:",
    round(complete_input_count / len(model_df) * 100, 2),
    "%"
)

Tüm girdileri dolu kayıt: 7899
En az bir girdisi eksik kayıt: 12473
Kullanılabilir oran: 38.77 %


In [7]:
missing_patterns = (
    model_df[input_columns]
    .isna()
    .value_counts()
    .reset_index(name="record_count")
)

display(missing_patterns.head(15))

,smiles_nodes,smiles_linkers,point_group,topology,record_count
0,False,False,False,True,9645
1,False,False,False,False,7899
2,True,True,False,True,2695
3,False,True,False,True,132
4,False,True,False,False,1


In [8]:
print("Farklı point group sayısı:")
print(model_df["point_group"].nunique(dropna=True))

display(
    model_df["point_group"]
    .value_counts(dropna=False)
    .head(20)
)

Farklı point group sayısı:
31


point_group
2/m      6947
-1       6197
1        2758
2         877
mmm       728
222       560
m         478
mm2       446
-3        209
422       139
3         138
4/m       115
32        112
6         109
4/mmm     103
-42m       66
-3m        65
622        47
3m         44
m-3m       41
Name: count, dtype: int64

In [9]:
print("Farklı topology sayısı:")
print(model_df["topology"].nunique(dropna=True))

display(
    model_df["topology"]
    .value_counts(dropna=False)
    .head(20)
)

Farklı topology sayısı:
211


topology
NaN    12472
pcu     1899
sql     1708
rna      701
hcb      361
dia      312
fcu      221
nbo      153
ttp      147
acs      138
bcu      136
kgd      124
fes      119
tsg      112
hxl       94
rtl       92
pts       88
fsc       85
bey       83
bex       74
Name: count, dtype: int64

In [10]:
display(
    model_df[
        [
            "smiles_nodes",
            "smiles_linkers",
        ]
    ].head(20)
)

,smiles_nodes,smiles_linkers
0,"['O', '[Ba]', '[Cu]']",['[O-]C=O']
1,NaN,NaN
2,['[Co]'],['[O-]C(=O)c1ccncc1']
3,['[Co]'],['[O-]C(=O)c1ccncc1']
4,['[Zn][Zn]'],"['[O-]C(=O)c1cccc(c1)c1nccs1', 'n1ccc(cc1)c1cc..."
5,"['[OH2][Ag][Ag][Ag][Ag][OH2]', '[OH2][Ag][Ag][...","['[O-]C(=O)C1=NN=C([CH]1)C(=O)[O-]', '[O-]C(=O..."
6,"['[Ag]1[Ag][Ag][Ag]1', '[Ag][Ag]']",['[O-]C(=O)c1nccnc1C(=O)[O-]']
7,['[OH2][Ag][Ag]'],['[O-]C(=O)c1nccnc1C(=O)[O-]']
8,['[Zn]'],"['Oc1cc(cc(c1)C(=O)[O-])C(=O)[O-]', 'n1ccc(cc1..."
9,['[Zn]'],"['Oc1cc(cc(c1)C(=O)[O-])C(=O)[O-]', 'n1ccc(cc1..."


In [11]:
print(
    "Tekrarlanan qmof_id sayısı:",
    model_df["qmof_id"].duplicated().sum()
)

print(
    "Tamamen tekrarlanan satır sayısı:",
    model_df.duplicated().sum()
)

Tekrarlanan qmof_id sayısı: 0
Tamamen tekrarlanan satır sayısı: 0


In [12]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = (
    PROCESSED_DIR
    / "forward_model_raw.csv"
)

model_df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8",
)

print("Kaydedildi:", OUTPUT_PATH)

Kaydedildi: c:\Users\ASUSTUF\Desktop\tubitak_internship\data\processed\forward_model_raw.csv
